# Gapfilling *Alteromonas macleodii* MIT1002 (iHS4156) for growth on mannuronate and galacturonate

**Goal:** enable the model to grow on **D-mannuronate** (`cpd01384`) and **D-galacturonate** (`cpd00280`) as sole carbon sources, consistent with experimental evidence.

## What the model already had (before gapfilling)

Both uronates are funnelled into central metabolism through **2-keto-3-deoxygluconate (KDG, `cpd00176`)**, which the model already converts to pyruvate + GAP via the Entner–Doudoroff lower branch:

KDG (`cpd00176`) → KDPG (`cpd02711`, `rxn01123`, kdgK) → pyruvate + glyceraldehyde-3-P (`rxn03884`, eda)

**Galacturonate** — the *entire* intracellular Ashwell (hexuronate) pathway was already present:

`cpd00280` D-galacturonate → `cpd00437` tagaturonate (`rxn01457`) → `cpd00608` altronate (`rxn01857`) → `cpd00176` KDG (`rxn01122`)

So galacturonate only needed an **extracellular metabolite + exchange + transporter**.

**Mannuronate** — `cpd01384` was absent, but its degradation product **D-mannonate (`cpd00403`)** was present, and `rxn03885` (mannonate dehydratase → KDG) was already in the model. The only missing internal link was **mannuronate reductase** (`rxn01776`/`rxn01777`): `cpd01384` ⇌ `cpd00403`.

## Changes made by this notebook

| Type | Mannuronate | Galacturonate |
|---|---|---|
| Extracellular metabolite | `cpd01384_e0` (new) | `cpd00280_e0` (new) |
| Cytoplasmic metabolite | `cpd01384_c0` (new) | `cpd00280_c0` (existed) |
| Exchange | `EX_cpd01384_e0` | `EX_cpd00280_e0` |
| Transporter (H⁺ symport) | `rxn_cpd01384_symport_c0` | `rxn_cpd00280_symport_c0` |
| Internal reaction(s) | `rxn01776` (NADH) + `rxn01777` (NADPH) reductase | none needed |

Transporters mirror the model's existing proton-symport convention (cf. galactose `rxn05566`). Metabolite formulas/charges come from the ModelSEED biochemistry DB (both uronates: `C6H9O7`, charge −1).

In [1]:
import cobra
from cobra import Metabolite, Reaction

model = cobra.io.read_sbml_model("model.xml")

# Where to write the gapfilled model. Change to e.g. "model_gapfilled.xml"
# if you would rather not overwrite the original (it is tracked in git).
OUTPUT_PATH = "model.xml"

print(model)
print("reactions:", len(model.reactions), "| metabolites:", len(model.metabolites))

iHS4156
reactions: 1417 | metabolites: 1234


In [2]:
mannuronate_id  = "cpd01384"   # D-Mannuronate
galacturonate_id = "cpd00280"  # D-Galacturonate
mannonate_id    = "cpd00403"   # D-Mannonate (already in model; mannuronate reductase product)

### 1. Add metabolites
`cpd00280_e0` copies the annotation of the existing cytoplasmic `cpd00280_c0`. Mannuronate metabolites use ModelSEED annotation for `cpd01384` (KEGG C02024).

In [3]:
def get_or_make(mid, name, formula, charge, compartment, annotation):
    if mid in model.metabolites:
        return model.metabolites.get_by_id(mid)
    met = Metabolite(mid, formula=formula, name=name, charge=charge, compartment=compartment)
    met.annotation = annotation
    model.add_metabolites([met])
    return met

manu_ann = {
    'sbo': 'SBO:0000247', 'seed.compound': 'cpd01384', 'kegg.compound': 'C02024',
    'metanetx.chemical': ['MNXM114059', 'MNXM2513'],
    'inchikey': 'AEMOLEFTQBMNLQ-VANFPWTGSA-M',
}
galur_ann = dict(model.metabolites.get_by_id("cpd00280_c0").annotation)  # reuse cytoplasmic annotation

manu_c  = get_or_make("cpd01384_c0", "D-Mannuronate [c0]",   "C6H9O7", -1, "c0", manu_ann)
manu_e  = get_or_make("cpd01384_e0", "D-Mannuronate [e0]",   "C6H9O7", -1, "e0", manu_ann)
galur_e = get_or_make("cpd00280_e0", "D-Galacturonate [e0]", "C6H9O7", -1, "e0", galur_ann)

for met in (manu_c, manu_e, galur_e):
    print(f"{met.id:14s} {met.formula:8s} charge={met.charge}  {met.name}")

cpd01384_c0    C6H9O7   charge=-1  D-Mannuronate [c0]
cpd01384_e0    C6H9O7   charge=-1  D-Mannuronate [e0]
cpd00280_e0    C6H9O7   charge=-1  D-Galacturonate [e0]


### 2. Add exchange reactions
Created reversible-capable but left **closed for uptake by default** (lower bound 0) so the base model behaviour is unchanged; uptake is opened explicitly in the growth tests / your simulations.

In [4]:
def add_exchange(met):
    rid = "EX_" + met.id
    if rid in model.reactions:
        return model.reactions.get_by_id(rid)
    r = Reaction(rid, name="Exchange for " + met.name.replace(" [e0]", ""))
    r.add_metabolites({met: -1.0})
    r.bounds = (0.0, 1000.0)
    r.annotation = {'sbo': 'SBO:0000627'}
    model.add_reactions([r])
    return r

ex_manu  = add_exchange(manu_e)
ex_galur = add_exchange(galur_e)
for r in (ex_manu, ex_galur):
    print(r.id, "|", r.reaction, "| bounds", r.bounds)

EX_cpd01384_e0 | cpd01384_e0 -->  | bounds (0.0, 1000.0)
EX_cpd00280_e0 | cpd00280_e0 -->  | bounds (0.0, 1000.0)


### 3. Add transporters (proton symport)
Uronate uptake in bacteria is typically H⁺-coupled (ExuT-type). Modelled as reversible proton symport, mirroring the model's galactose transporter `rxn05566`.

In [5]:
H_e = model.metabolites.get_by_id("cpd00067_e0")
H_c = model.metabolites.get_by_id("cpd00067_c0")

def add_symport(rid, name, sub_e, sub_c):
    if rid in model.reactions:
        return model.reactions.get_by_id(rid)
    r = Reaction(rid, name=name)
    r.bounds = (-1000.0, 1000.0)
    r.add_metabolites({H_e: -1.0, sub_e: -1.0, H_c: 1.0, sub_c: 1.0})
    r.annotation = {'sbo': 'SBO:0000185'}
    model.add_reactions([r])
    return r

t_manu  = add_symport("rxn_cpd01384_symport_c0", "D-Mannuronate transport in via proton symport [c0]",   manu_e,  manu_c)
t_galur = add_symport("rxn_cpd00280_symport_c0", "D-Galacturonate transport in via proton symport [c0]", galur_e, model.metabolites.get_by_id("cpd00280_c0"))
for r in (t_manu, t_galur):
    print(r.id, "|", r.reaction)

rxn_cpd01384_symport_c0 | cpd00067_e0 + cpd01384_e0 <=> cpd00067_c0 + cpd01384_c0
rxn_cpd00280_symport_c0 | cpd00067_e0 + cpd00280_e0 <=> cpd00067_c0 + cpd00280_c0


### 4. Add the missing internal reaction: mannuronate reductase

`cpd01384` (mannuronate) ⇌ `cpd00403` (mannonate). ModelSEED `rxn01776` (NAD⁺) and `rxn01777` (NADP⁺) are both reversible; degradation runs in the mannuronate→mannonate direction. Mannonate then enters the existing network via `rxn03885` (mannonate dehydratase → KDG).

Galacturonate needs **no** new internal reactions — its full pathway to KDG already exists.

In [6]:
mannonate = model.metabolites.get_by_id("cpd00403_c0")
NAD   = model.metabolites.get_by_id("cpd00003_c0"); NADH  = model.metabolites.get_by_id("cpd00004_c0")
NADP  = model.metabolites.get_by_id("cpd00006_c0"); NADPH = model.metabolites.get_by_id("cpd00005_c0")

def add_internal(rid, name, stoich, seedid):
    fid = rid + "_c0"
    if fid in model.reactions:
        return model.reactions.get_by_id(fid)
    r = Reaction(fid, name=name)
    r.bounds = (-1000.0, 1000.0)
    r.add_metabolites(stoich)
    r.annotation = {'sbo': 'SBO:0000176', 'seed.reaction': seedid}
    model.add_reactions([r])
    return r

# rxn01776:  NAD  + D-mannonate  <=> NADH  + H+ + D-mannuronate
# rxn01777:  NADP + D-mannonate  <=> NADPH + H+ + D-mannuronate
r76 = add_internal("rxn01776", "D-Mannonate:NAD+ 6-oxidoreductase",
                   {NAD: -1.0, mannonate: -1.0, NADH: 1.0, H_c: 1.0, manu_c: 1.0}, "rxn01776")
r77 = add_internal("rxn01777", "D-Mannonate:NADP+ 6-oxidoreductase",
                   {NADP: -1.0, mannonate: -1.0, NADPH: 1.0, H_c: 1.0, manu_c: 1.0}, "rxn01777")
for r in (r76, r77):
    print(r.id, "|", r.reaction)

rxn01776_c0 | cpd00003_c0 + cpd00403_c0 <=> cpd00004_c0 + cpd00067_c0 + cpd01384_c0
rxn01777_c0 | cpd00006_c0 + cpd00403_c0 <=> cpd00005_c0 + cpd00067_c0 + cpd01384_c0


### 5. Verify mass & charge balance of every new reaction
An empty dict means the reaction is fully balanced.

In [7]:
new_rxns = [ex_manu, ex_galur, t_manu, t_galur, r76, r77]
for r in new_rxns:
    imb = r.check_mass_balance()
    # exchanges are open boundary reactions (intentionally unbalanced); skip those
    tag = "(boundary)" if r.boundary else ("BALANCED" if not imb else f"IMBALANCE: {imb}")
    print(f"{r.id:28s} {tag}")

EX_cpd01384_e0               (boundary)
EX_cpd00280_e0               (boundary)
rxn_cpd01384_symport_c0      BALANCED
rxn_cpd00280_symport_c0      BALANCED
rxn01776_c0                  BALANCED
rxn01777_c0                  BALANCED


### 6. Verify growth on each substrate as the *sole* carbon source
The model's default medium is carbon-free (minerals + thiamin only), so adding a single uronate exchange isolates growth on that carbon source. Tests use a temporary `with model:` context so the saved model keeps its default (closed) uptake bounds.

In [8]:
def growth_on(ex_id, label):
    with model:
        medium = model.medium
        medium[ex_id] = 10.0          # open uptake of the test carbon source
        model.medium = medium
        g = model.slim_optimize()
    print(f"{label:32s} biomass = {g:.4f}" if g == g else f"{label:32s} NO GROWTH (infeasible)")
    return g

g_no   = model.slim_optimize()
print(f"{'no carbon source (control)':32s} " + ("NO GROWTH (infeasible)" if g_no != g_no else f"biomass = {g_no:.4f}"))
g_manu  = growth_on("EX_cpd01384_e0", "D-mannuronate sole carbon")
g_galur = growth_on("EX_cpd00280_e0", "D-galacturonate sole carbon")

assert g_manu  and g_manu  > 1e-6, "mannuronate growth failed"
assert g_galur and g_galur > 1e-6, "galacturonate growth failed"
print("\nBoth substrates support growth.")

no carbon source (control)       NO GROWTH (infeasible)
D-mannuronate sole carbon        biomass = 0.6453
D-galacturonate sole carbon      biomass = 0.6453

Both substrates support growth.


### 7. Save the gapfilled model

In [9]:
cobra.io.write_sbml_model(model, OUTPUT_PATH)
print("Saved:", OUTPUT_PATH)
print("reactions:", len(model.reactions), "| metabolites:", len(model.metabolites))

Saved: model.xml
reactions: 1423 | metabolites: 1237
